# Graph-Based Warehouse Slotting Optimization

**Objective**: Reduce picking travel time by grouping products with high "purchase affinity" into shared zones using Graph Theory and Node2Vec embeddings.

### 1. Theoretical Framework

#### The Bipartite Network
We model the warehouse data as a Bipartite Graph $G = (U, V, E)$, where:
- $U$ is the set of **Orders**.
- $V$ is the set of **Products**.
- An edge exists if Product $v \in V$ appears in Order $u \in U$.

#### Jaccard Similarity Projection
To find product-product relationships, we project $G$ onto $V$. The weight $w$ between two products $i$ and $j$ is calculated using the **Jaccard Index**:

$$J(i, j) = \frac{|N(i) \cap N(j)|}{|N(i) \cup N(j)|}$$

Where $N(x)$ is the set of orders containing product $x$. This prevents "popular" products from dominating the graph unfairly.

In [ ]:
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import community as community_louvain
import os

# Styling
plt.style.use('seaborn-v0_8-muted')
sns.set_palette("viridis")
random.seed(42)
np.random.seed(42)

## 2. Data Preparation & Enrichment
We load the Olist dataset and translate category names to English for better interpretability.

In [ ]:
data_dir = 'data'
order_items = pd.read_csv(os.path.join(data_dir, 'olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(data_dir, 'olist_products_dataset.csv'))
translation = pd.read_csv(os.path.join(data_dir, 'product_category_name_translation.csv'))

# Map categories to English
cat_map = dict(zip(translation['product_category_name'], translation['product_category_name_english']))
products['category'] = products['product_category_name'].map(cat_map).fillna('other')

print(f"Loaded {len(order_items)} items. mapping {len(cat_map)} categories.")
order_items.head()

## 3. Graph Construction
### 3.1 Bipartite View
Before fully processing, let's visualize the structural relationship between a few orders and their products.

In [ ]:
# Sample 5 orders with multiple products
multi_item_orders = order_items.groupby('order_id').filter(lambda x: len(x) > 1)['order_id'].unique()[:5]
sample_df = order_items[order_items['order_id'].isin(multi_item_orders)]

BG_sample = nx.Graph()
BG_sample.add_nodes_from(sample_df['order_id'], bipartite=0)
BG_sample.add_nodes_from(sample_df['product_id'], bipartite=1)
BG_sample.add_edges_from(zip(sample_df['order_id'], sample_df['product_id']))

pos = nx.bipartite_layout(BG_sample, multi_item_orders)
plt.figure(figsize=(10, 6))
nx.draw(BG_sample, pos, with_labels=False, node_color=['skyblue' if d['bipartite']==0 else 'orange' for n,d in BG_sample.nodes(data=True)], node_size=100)
plt.title("Bipartite Topology: Orders (Blue) connected to Products (Orange)")
plt.show()

### 3.2 Product-Product Projection
We now project the full graph into a Product-Product Network weighted by co-occurrence affinity.

In [ ]:
# Filter for stability
product_counts = order_items['product_id'].value_counts()
stable_products = product_counts[product_counts > 1].index
filtered_items = order_items[order_items['product_id'].isin(stable_products)]

B = nx.Graph()
B.add_nodes_from(filtered_items['order_id'].unique(), bipartite=0)
B.add_nodes_from(filtered_items['product_id'].unique(), bipartite=1)
B.add_edges_from(zip(filtered_items['order_id'], filtered_items['product_id']))

product_nodes = [n for n, d in B.nodes(data=True) if d['bipartite'] == 1]
P = bipartite.weighted_projected_graph(B, product_nodes)

# Apply Jaccard weights
weights = []
for u, v, d in P.edges(data=True):
    intersection = d['weight']
    union = B.degree(u) + B.degree(v) - intersection
    d['weight'] = intersection / union
    weights.append(d['weight'])

plt.figure(figsize=(8, 4))
sns.histplot(weights, bins=30, kde=True, color='purple')
plt.title("Distribution of Jaccard Affinity Weights between Products")
plt.xlabel("Jaccard Similarity Score")
plt.show()

## 4. Latent Space Learning (Node2Vec)
Traditional clustering only sees direct connections. **Node2Vec** uses random walks to capture higher-order relationships—identifying products that are "functionally similar" even if they haven't appeared in the same order yet.

In [ ]:
def run_random_walks(G, walk_len=30, n_walks=10):
    all_walks = []
    nodes = list(G.nodes())
    for _ in range(n_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = [node]
            while len(walk) < walk_len:
                curr = walk[-1]
                nbrs = list(G.neighbors(curr))
                if not nbrs: break
                walk.append(random.choice(nbrs))
            all_walks.append([str(x) for x in walk])
    return all_walks

print("Learning community structures via random walks...")
walks = run_random_walks(P)
model = Word2Vec(walks, vector_size=64, window=5, min_count=1, sg=1, workers=4)

embeddings = {node: model.wv[str(node)] for node in P.nodes()}
print(f"Generated vectors for {len(embeddings)} products.")

## 5. Zoning Analysis
We group products into 12 clusters (Warehouse Zones) and analyze their category composition.

In [ ]:
pids = list(embeddings.keys())
X = np.array([embeddings[p] for p in pids])
kmeans = KMeans(n_clusters=12, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(X)

product_data = pd.DataFrame({'product_id': pids, 'zone': cluster_ids})
product_data = product_data.merge(products[['product_id', 'category']], on='product_id')

# Visualize top category in each zone
plt.figure(figsize=(14, 6))
zone_counts = product_data.groupby(['zone', 'category']).size().reset_index(name='count')
top_cats = zone_counts.sort_values(['zone', 'count'], ascending=[True, False]).groupby('zone').head(1)

sns.barplot(data=top_cats, x='zone', y='count', hue='category', dodge=False)
plt.title("Dominant Product Categories per Warehouse Zone")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Primary Category")
plt.show()

## 6. Simulation & Efficiency Benchmark
Comparing travel distances between the **Random Baseline** and our **Graph-Optimized Zoning**.

In [ ]:
class WarehouseSim:
    def __init__(self, p_data):
        self.p_data = p_data
        self.p_map = dict(zip(p_data['product_id'], p_data['zone']))
        self.grid_side = int(np.ceil(np.sqrt(len(p_data)))) + 5
        
    def generate_layouts(self):
        # Optimized: Grouped by zone
        opt_pids = sorted(self.p_map.keys(), key=lambda x: self.p_map[x])
        opt_layout = {pid: (i % self.grid_side, i // self.grid_side) for i, pid in enumerate(opt_pids)}
        
        # Baseline: Random scramble
        base_pids = list(self.p_map.keys())
        random.shuffle(base_pids)
        base_layout = {pid: (i % self.grid_side, i // self.grid_side) for i, pid in enumerate(base_pids)}
        
        return base_layout, opt_layout

    def run(self, orders, layouts, n_samples=1000):
        results = {'baseline': [], 'optimized': []}
        sampled = random.sample(orders, min(n_samples, len(orders)))
        
        for order in sampled:
            for mode, layout in layouts.items():
                dist = 0; curr = (0,0)
                order_pids = [p for p in order if p in layout]
                if not order_pids: continue
                for p in order_pids:
                    target = layout[p]
                    dist += abs(curr[0]-target[0]) + abs(curr[1]-target[1])
                    curr = target
                dist += abs(curr[0]) + abs(curr[1]) # Return to dock
                results[mode].append(dist)
        return results

sim = WarehouseSim(product_data)
layouts = dict(zip(['baseline', 'optimized'], sim.generate_layouts()))
all_orders = order_items.groupby('order_id')['product_id'].apply(list).tolist()
sim_results = sim.run(all_orders, layouts)

plt.figure(figsize=(10, 5))
sns.kdeplot(sim_results['baseline'], shade=True, label='Baseline (Random)')
sns.kdeplot(sim_results['optimized'], shade=True, label='Optimized (Graph-Zoning)')
plt.title("Picking Travel Distance Distribution: Baseline vs Optimized")
plt.xlabel("Total Manhattan Distance per Order")
plt.legend()
plt.show()

print(f"Avg Distance Reduction: {100*(1 - np.mean(sim_results['optimized'])/np.mean(sim_results['baseline'])):.1f}%")

## 7. Conclusions
By applying Graph Topology and embedding-based clustering, we effectively concentrated co-purchased items into physical proximity. The distribution plot above shows a clear shift to the left for the optimized layout, indicating a significant reduction in human/robot travel time.